In [9]:
from mpi4py import MPI
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dolfinx import fem as fe
import os
from itertools import product
from tqdm import tqdm
import time
import seaborn as sns
 
from data_io import save_pickle, load_pickle
from metrics import calculate_analysis_metrics
from dca_utils import*
 
from fourd_var import run_assimilation, setup_data_assimilation, run_data_assimilation
from plotting import plot_mixed_function, plot_comparison1, plot_comparison2

sns.set_palette("bright")
plt.style.use("mystyle1.mplstyle")


In [10]:
def run_parameter_crossval(pickle_path, problem_params, prob, solver_params, 
                       obs_space_freq, obs_time_freq, final_time,
                       obs_std_values, inflation_factor_values, station_ids, 
                       output_dir='da_output', verbose=True, var_type='dci_wme'):
    """
    Run parameter cross validation over observation standard deviation and inflation factor values.
    """
    
    def run_single_experiment(obs_std, inflation_factor):
        """Run a single experiment and return results."""
        exp_name = f"obs_std_{obs_std}_inflation_{inflation_factor}"
        
        try:
            # Setup and run assimilation
            result = setup_data_assimilation(
                pickle_path=pickle_path,
                problem_params=problem_params.copy(),
                prob=prob,
                obs_std=obs_std,
                obs_space_freq=obs_space_freq,
                obs_time_freq=obs_time_freq,
                station_ids=station_ids,
                final_time=final_time,
                inflation_factor=inflation_factor,
                print_setup=True
            )
            save_pickle("setup_result.pkl", result)
            
            analysis, run_bathy = run_assimilation(
                result['problem_params'], solver_params, result['stations'],
                result['y_obs'], result['obs_per_window'], result['obs_time_indices'],
                result['H'], result['covs'], result['hb'], 'sloped_beach',
                cost_function_type=var_type
            )
            
        
            var_rmse, var_misfit = calculate_analysis_metrics(
                analysis_name=var_type,
                analysis_data=analysis,
                save_first=True,
            )

            # Save results
            output_filename = f'{var_type}_analysis_{exp_name}.pkl'
            save_pickle(output_filename, analysis)
            
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'analysis': analysis,
                'setup_result': result,
                'output_file': output_filename
            }
            
        except SystemExit as e:
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'error': f'Solver convergence failure (SystemExit: {e.code})',
                'error_type': 'convergence_failure'
            }
        except Exception as e:
            return {
                'obs_std': obs_std,
                'inflation_factor': inflation_factor,
                'error': str(e),
                'error_type': 'general_exception'
            }
    
    def print_progress(exp_count, total, exp_name, start_time, result):
        """Print experiment progress."""
        elapsed = time.time() - start_time
        if 'error' in result:
            print(f"  ✗ Failed after {elapsed:.2f}s: {result.get('error', 'Unknown error')}")
        else:
            print(f"  ✓ Completed in {elapsed:.2f}s - Saved to {result['output_file']}")
    
    def print_summary(all_results):
        """Print final summary statistics."""
        total = len(all_results)
        successful = sum(1 for r in all_results.values() if 'error' not in r)
        convergence_failures = sum(1 for r in all_results.values() 
                                 if r.get('error_type') == 'convergence_failure')
        other_failures = total - successful - convergence_failures
        
        print(f"\nParameter cross validation completed!")
        print(f"Successful: {successful}/{total}")
        if convergence_failures > 0:
            print(f"Convergence failures: {convergence_failures}")
        if other_failures > 0:
            print(f"Other failures: {other_failures}")
        
        if convergence_failures > 0:
            print(f"\nConvergence failure parameters:")
            for result in all_results.values():
                if result.get('error_type') == 'convergence_failure':
                    print(f"  obs_std={result['obs_std']}, inflation_factor={result['inflation_factor']}")
    
    # Main execution
    os.makedirs(output_dir, exist_ok=True)
    all_results = {}
    param_combinations = list(product(obs_std_values, inflation_factor_values))
    
    if verbose:
        print(f"Starting {len(param_combinations)} experiments")
        print(f"obs_std: {obs_std_values}")
        print(f"inflation_factor: {inflation_factor_values}")
        print("=" * 80)
    
    # Run all experiments
    for i, (obs_std, inflation_factor) in enumerate(param_combinations, 1):
        exp_name = f"obs_std_{obs_std}_inflation_{inflation_factor}"
        
        if verbose:
            print(f"\nExperiment {i}/{len(param_combinations)}: {exp_name}")
            start_time = time.time()
        
        result = run_single_experiment(obs_std, inflation_factor)
        all_results[exp_name] = result
        
        if verbose:
            print_progress(i, len(param_combinations), exp_name, start_time, result)
    
    # Save summary and print results
    save_pickle('parameter_crossval_summary.pkl', all_results)
    
    if verbose:
        print("=" * 80)
        print_summary(all_results)
    
    return all_results

In [ ]:
        # 'obs_std_values': [1.0, 1.5, 2.0],
        # 'inflation_factor_values': [4.0, 8.0, 12.0],

In [42]:
if __name__ == "__main__":
    # Configuration
    CONFIG = {
        'obs_std_values': [0.01],
        'inflation_factor_values': [1.0],
        'final_time': Time.ONE_DAY.seconds,
        'window_size': Time.ONE_HOUR.seconds,
        'dt': 600,  # 10 minutes
        'run_true': True,
    }
    
    # Problem parameters
    problem_params = {
        'dt': CONFIG['dt'],
        't': 0,
        't_final': CONFIG['final_time'],
        'num_steps': int(np.ceil(CONFIG['final_time'] / CONFIG['dt'])),
        'num_windows': CONFIG['final_time'] // CONFIG['window_size'],
        'fric_law': 'linear',
        'alpha': 2.0 * np.pi / Time.TWELVE_HOURS.seconds,
        'sol_var': 'h'
    }
    
    # Solver parameters
    solver_params = {
        "rtol": 1e-5,
        "atol": 1e-6, 
        "max_it": 10,
        "relaxation_parameter": 1.0,
        "ksp_type": "gmres",
        "pc_type": "ilu",
        "ksp_ErrorIfNotConverged": False
    }
    
    # Station configuration
    # station_ids = {
    #     'method': 'region',
    #     'params': {
    #         'bounds': {'x': (1000, 6000.0), 'y': (1000, 6000)},
    #         'criteria': 'center'
    #     }
    # }
    stat_ids = [21, 64, 81, 103, 136]
    station_ids = {
        'method': 'indices', 
        'params': stat_ids
    }
    # Setup problem and generate true signal
    prob, solver = create_problem_solver(problem_params, "sloped_beach", true_signal=True, verbose=False)
    
    if CONFIG['run_true']:
        assert problem_params['num_steps'] == int(np.ceil(problem_params['t_final'] / problem_params['dt']))
        true_solver = get_true_signal(solver, 'sloped_beach', solver_params, 1)

    
    # Run parameter cross validation
    results = run_parameter_crossval(
        pickle_path='true_signal.pkl',
        problem_params=problem_params,
        prob=prob,
        solver_params=solver_params,
        obs_space_freq=2,  # Legacy parameter, will be removed
        obs_time_freq=1,
        final_time=CONFIG['final_time'],
        obs_std_values=CONFIG['obs_std_values'],
        inflation_factor_values=CONFIG['inflation_factor_values'],
        station_ids=station_ids,
        output_dir='da_output',
        verbose=True,
        var_type='dci_wme'
    )
    
    # Print summary
    def print_results_summary(results):
        """Print a clean summary of experiment results."""
        print("\nParameter sweep results summary:")
        print("-" * 40)
        
        for exp_name, output in results.items():
            if 'error' not in output:
                print(f"✓ {exp_name}: SUCCESS")
            elif output.get('error_type') == 'convergence_failure':
                print(f"✗ {exp_name}: CONVERGENCE FAILURE")
            else:
                print(f"✗ {exp_name}: ERROR - {output['error'][:50]}...")
        
        # Convergence failure recommendations
        convergence_failures = sum(1 for r in results.values() 
                                 if r.get('error_type') == 'convergence_failure')
        
        if convergence_failures > 0:
            print(f"\n⚠️  {convergence_failures} experiments failed due to convergence issues.")
            print("Consider adjusting:")
            print("• obs_std and inflation_factor values")
            print("• Solver tolerances or max iterations")
            print("• Observation setup parameters")
    
    print_results_summary(results)

Starting 1 experiments
obs_std: [0.01]
inflation_factor: [1.0]

Experiment 1/1: obs_std_0.01_inflation_1.0
Selected 5 stations using method 'indices'
Station cells: [ 21  64  81 103 136]
DATA ASSIMILATION EXPERIMENT SETUP

Input Parameters:
  Pickle path: true_signal.pkl
  Observation std deviation: 0.01
  Observation time frequency: 1
  Final time: 86400
  Inflation factor: 1.0

Problem Parameters:
  dt: 600
  t: 0
  t_final: 86400
  num_steps: 6
  num_windows: 24
  fric_law: mannings
  alpha: 0.0001454441043328608
  sol_var: h

Calculated Dimensions:
  State dimension: 1296
  Observation dimension: 5
  Total time steps: 145
  Observations per window: 6
  Number of observation obs_stations: 5

Matrix Information:
  Observation matrix H shape: (5, 1296)
  Background covariance B shape: (1296, 1296)
  Observation covariance R shape: (5, 5)
  Predicted covariance L shape: (5, 5)

Observation Setup:
  Number of observation time indices: 145
  Observation spatial indices: [ 21  64  81 103 

Processing windows:  29%|██▉       | 7/24 [00:23<01:14,  4.36s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 2.660447e+03
  Iterations: 1
  Function evaluations: 31
  Gradient norm at solution: 1.328201e-01

------------------------------------------------------------



Processing windows: 100%|██████████| 24/24 [01:22<00:00,  3.45s/window]

DCI_WME RMSE: 0.2906545512, DCI_WME, Relative Misfit: 0.1187
  ✓ Completed in 83.02s - Saved to dci_wme_analysis_obs_std_0.01_inflation_1.0.pkl

Parameter cross validation completed!
Successful: 1/1

Parameter sweep results summary:
----------------------------------------
✓ obs_std_0.01_inflation_1.0: SUCCESS


In [ ]:
if __name__ == "__main__":
    # Configuration
    CONFIG = {
        'obs_std_values': [0.01, 0.05],
        'inflation_factor_values': [0.2, 0.4, 0.5, 1.0],
        'final_time': Time.SEVEN_DAYS.seconds,
        'window_size': Time.ONE_HOUR.seconds,
        'dt': 600,  # 10 minutes
        'run_true': True,
    }
    
    # Problem parameters
    problem_params = {
        'dt': CONFIG['dt'],
        't': 0,
        't_final': CONFIG['final_time'],
        'num_steps': int(np.ceil(CONFIG['final_time'] / CONFIG['dt'])),
        'num_windows': CONFIG['final_time'] // CONFIG['window_size'],
        'fric_law': 'linear',
        'alpha': 2.0 * np.pi / Time.TWELVE_HOURS.seconds,
        'sol_var': 'h'
    }
    
    # Solver parameters
    solver_params = {
        "rtol": 1e-5,
        "atol": 1e-6, 
        "max_it": 10,
        "relaxation_parameter": 1.0,
        "ksp_type": "gmres",
        "pc_type": "ilu",
        "ksp_ErrorIfNotConverged": False
    }
    
    # Station configuration
    # station_ids = {
    #     'method': 'region',
    #     'params': {
    #         'bounds': {'x': (1000, 6000.0), 'y': (1000, 6000)},
    #         'criteria': 'center'
    #     }
    # }
    stat_ids = [21, 64, 81, 103, 136]
    station_ids = {
        'method': 'indices', 
        'params': stat_ids
    }
    # Setup problem and generate true signal
    prob, solver = create_problem_solver(problem_params, "sloped_beach", true_signal=True, verbose=False)
    
    if CONFIG['run_true']:
        assert problem_params['num_steps'] == int(np.ceil(problem_params['t_final'] / problem_params['dt']))
        true_solver = get_true_signal(solver, 'sloped_beach', solver_params, 1)

    
    # Run parameter cross validation
    results = run_parameter_crossval(
        pickle_path='true_signal.pkl',
        problem_params=problem_params,
        prob=prob,
        solver_params=solver_params,
        obs_space_freq=2,  # Legacy parameter, will be removed
        obs_time_freq=1,
        final_time=CONFIG['final_time'],
        obs_std_values=CONFIG['obs_std_values'],
        inflation_factor_values=CONFIG['inflation_factor_values'],
        station_ids=station_ids,
        output_dir='da_output',
        verbose=True,
        var_type='bayes'
    )
    
    # Print summary
    def print_results_summary(results):
        """Print a clean summary of experiment results."""
        print("\nParameter sweep results summary:")
        print("-" * 40)
        
        for exp_name, output in results.items():
            if 'error' not in output:
                print(f"✓ {exp_name}: SUCCESS")
            elif output.get('error_type') == 'convergence_failure':
                print(f"✗ {exp_name}: CONVERGENCE FAILURE")
            else:
                print(f"✗ {exp_name}: ERROR - {output['error'][:50]}...")
        
        # Convergence failure recommendations
        convergence_failures = sum(1 for r in results.values() 
                                 if r.get('error_type') == 'convergence_failure')
        
        if convergence_failures > 0:
            print(f"\n⚠️  {convergence_failures} experiments failed due to convergence issues.")
            print("Consider adjusting:")
            print("• obs_std and inflation_factor values")
            print("• Solver tolerances or max iterations")
            print("• Observation setup parameters")
    
    print_results_summary(results)

Starting 8 experiments
obs_std: [0.01, 0.05]
inflation_factor: [0.2, 0.4, 0.5, 1.0]

Experiment 1/8: obs_std_0.01_inflation_0.2
Selected 5 stations using method 'indices'
Station cells: [ 21  64  81 103 136]
DATA ASSIMILATION EXPERIMENT SETUP

Input Parameters:
  Pickle path: true_signal.pkl
  Observation std deviation: 0.01
  Observation time frequency: 1
  Final time: 604800
  Inflation factor: 0.2

Problem Parameters:
  dt: 600
  t: 0
  t_final: 604800
  num_steps: 6
  num_windows: 168
  fric_law: mannings
  alpha: 0.0001454441043328608
  sol_var: h

Calculated Dimensions:
  State dimension: 1296
  Observation dimension: 5
  Total time steps: 1009
  Observations per window: 6
  Number of observation obs_stations: 5

Matrix Information:
  Observation matrix H shape: (5, 1296)
  Background covariance B shape: (1296, 1296)
  Observation covariance R shape: (5, 5)
  Predicted covariance L shape: (5, 5)

Observation Setup:
  Number of observation time indices: 1009
  Observation spatial 

Processing windows:   0%|          | 0/168 [00:04<?, ?window/s]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 1.739434e+01
  Iterations: 0
  Function evaluations: 21
  Gradient norm at solution: 1.113833e+01

------------------------------------------------------------



Processing windows:   8%|▊         | 14/168 [00:49<09:46,  3.81s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 4.032536e+04
  Iterations: 2
  Function evaluations: 34
  Gradient norm at solution: 2.153640e-02

------------------------------------------------------------



...

Processing windows:  20%|██        | 34/168 [01:27<06:03,  2.71s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 6.682294e+03
  Iterations: 0
  Function evaluations: 16
  Gradient norm at solution: 3.908842e-01

------------------------------------------------------------



Processing windows:  21%|██        | 35/168 [01:53<06:27,  2.91s/window]

Optimization failed:


Optimization completed:
  Success: False
  Status: 2
  Message: ABNORMAL: 
  Final cost: 1.179294e+03
  Iterations: 0
  Function evaluations: 15
  Gradient norm at solution: 5.220114e-01

------------------------------------------------------------



Processing windows:  35%|███▌      | 59/168 [03:11<06:32,  3.60s/window]

In [41]:
result = load_pickle('setup_result.pkl')
wme_results = analyze_error_statistics(CONFIG['obs_std_values'], CONFIG['inflation_factor_values'], result, 
                            analysis_type='dc_wme')

Error Statistics for DC_WME Analysis
Obs Std    Inflation    RMSE            Misfit         
------------------------------------------------------------
0.010      0.500        ERROR/NOT FOUND N/A            
0.010      1.000        ERROR/NOT FOUND N/A            
0.010      1.500        ERROR/NOT FOUND N/A            
------------------------------------------------------------


In [38]:
result = load_pickle('setup_result.pkl')
wme_results = analyze_error_statistics(CONFIG['obs_std_values'], CONFIG['inflation_factor_values'], result, 
                            analysis_type='bayes')

Error Statistics for BAYES Analysis
Obs Std    Inflation    RMSE            Misfit         
------------------------------------------------------------
0.010      0.500        0.314023        0.142864       
0.010      1.000        0.317426        0.142910       
0.010      1.500        0.318962        0.143205       
------------------------------------------------------------


In [ ]:
# Create DataFrame from dictionary directly, then reset index
df = pd.DataFrame.from_dict(wme_results, orient='index')
df.index.names = ['obs_std', 'inflation_factor']
df = df.reset_index()
df.index.name = "BAYES: Window = 1 Hour"
df.to_csv('da_output/bayes_one_hour_results.csv', index=True)

In [ ]:
result = setup_data_assimilation(
    pickle_path='true_signal.pkl',
    problem_params=problem_params,
    prob=prob,
    obs_std=1.5,
    obs_space_freq=2,
    obs_time_freq=1,
    final_time=final_time,
    inflation_factor=8.0,  
    print_setup=True
)